In [1]:
# This code will plot an histogram based on the ocurrence rate of MM waves with respect to the hemisphere of the comet

In [2]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_coordinates=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_coordinates.dt.year == int(year)
    include = data_plasma[mask]
    time_coordinates2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_coordinates2.dt.month == int(month)
    include2 = include[mask2]
    time_coordinates3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_coordinates3.dt.day == int(day)
    include3=include2[mask3]

    time_coordinates4=pd.to_datetime(include3['t_utc'])
    
  
    return time_coordinates4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

In [3]:
#Importing data

path="C:/Users/Ariel/Desktop/ESA/position_CK.txt"
titles=['Time', 'x', 'y', 'z']
data= pd.read_table(str(path), header=None, names=titles, parse_dates=['Time']) #, sep='\s+'
data=data.iloc[np.arange(1,len(data)),:] #delete the first row
data=data.reset_index(drop=True)

data['Time']= pd.to_datetime(data['Time'])
#----------------------------------------
#print(data['Time'][0])
#print(data)

path2="C:/Users/Ariel/Desktop/JUPYTER/Resultados/MMWavesRosettaMission"
titles2=['Beginning','End','Index1','Index2','Pearson']
data2= pd.read_table(path2, header=None, names=titles2 ,sep='\t')#,engine='python') #sep='\t'
data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
data2=data2.reset_index(drop=True)
data2['Beginning']= pd.to_datetime(data2['Beginning'])
data2['End']= pd.to_datetime(data2['End'])
print(data2)


for i in np.arange(0,len(data2)):
    
    data2['Beginning'][i]=data2['Beginning'][i].replace(microsecond=0, second=0)
    data2['End'][i]=data2['End'][i].replace(microsecond=0, second=0)

print(data2)

C:\Users\Ariel\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3338: DtypeWarning: Columns (1,2,3) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


              Beginning                 End   Index1   Index2  \
0   2014-12-14 09:55:30 2014-12-14 09:56:07  35745.0  35752.0   
1   2014-12-14 10:39:07 2014-12-14 10:39:38  38362.0  38363.0   
2   2014-12-15 18:56:06 2014-12-15 18:56:36  68181.0  68181.0   
3   2014-12-18 22:14:20 2014-12-18 22:15:00  80075.0  80085.0   
4   2014-12-24 01:04:58 2014-12-24 01:05:30   3913.0   3915.0   
..                  ...                 ...      ...      ...   
560 2016-02-28 00:22:41 2016-02-28 00:23:21   1376.0   1386.0   
561 2016-02-28 06:53:13 2016-02-28 06:53:48  24808.0  24813.0   
562 2016-02-28 11:27:40 2016-02-28 11:28:13  41275.0  41278.0   
563 2016-02-28 11:33:27 2016-02-28 11:34:02  41622.0  41627.0   
564 2016-02-28 11:33:48 2016-02-28 11:34:20  41643.0  41645.0   

                 Pearson  
0    -0.7731610082993551  
1    -0.7194916719216243  
2    -0.7126112073191414  
3    -0.7372360235593105  
4    -0.8271626975988577  
..                   ...  
560  -0.8281193006894146  
561

<ipython-input-3-b4e23f757c63>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2['Beginning'][i]=data2['Beginning'][i].replace(microsecond=0, second=0)
<ipython-input-3-b4e23f757c63>:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2['End'][i]=data2['End'][i].replace(microsecond=0, second=0)


              Beginning                 End   Index1   Index2  \
0   2014-12-14 09:55:00 2014-12-14 09:56:00  35745.0  35752.0   
1   2014-12-14 10:39:00 2014-12-14 10:39:00  38362.0  38363.0   
2   2014-12-15 18:56:00 2014-12-15 18:56:00  68181.0  68181.0   
3   2014-12-18 22:14:00 2014-12-18 22:15:00  80075.0  80085.0   
4   2014-12-24 01:04:00 2014-12-24 01:05:00   3913.0   3915.0   
..                  ...                 ...      ...      ...   
560 2016-02-28 00:22:00 2016-02-28 00:23:00   1376.0   1386.0   
561 2016-02-28 06:53:00 2016-02-28 06:53:00  24808.0  24813.0   
562 2016-02-28 11:27:00 2016-02-28 11:28:00  41275.0  41278.0   
563 2016-02-28 11:33:00 2016-02-28 11:34:00  41622.0  41627.0   
564 2016-02-28 11:33:00 2016-02-28 11:34:00  41643.0  41645.0   

                 Pearson  
0    -0.7731610082993551  
1    -0.7194916719216243  
2    -0.7126112073191414  
3    -0.7372360235593105  
4    -0.8271626975988577  
..                   ...  
560  -0.8281193006894146  
561

In [4]:
print(len(data2))
print(data2)

565
              Beginning                 End   Index1   Index2  \
0   2014-12-14 09:55:00 2014-12-14 09:56:00  35745.0  35752.0   
1   2014-12-14 10:39:00 2014-12-14 10:39:00  38362.0  38363.0   
2   2014-12-15 18:56:00 2014-12-15 18:56:00  68181.0  68181.0   
3   2014-12-18 22:14:00 2014-12-18 22:15:00  80075.0  80085.0   
4   2014-12-24 01:04:00 2014-12-24 01:05:00   3913.0   3915.0   
..                  ...                 ...      ...      ...   
560 2016-02-28 00:22:00 2016-02-28 00:23:00   1376.0   1386.0   
561 2016-02-28 06:53:00 2016-02-28 06:53:00  24808.0  24813.0   
562 2016-02-28 11:27:00 2016-02-28 11:28:00  41275.0  41278.0   
563 2016-02-28 11:33:00 2016-02-28 11:34:00  41622.0  41627.0   
564 2016-02-28 11:33:00 2016-02-28 11:34:00  41643.0  41645.0   

                 Pearson  
0    -0.7731610082993551  
1    -0.7194916719216243  
2    -0.7126112073191414  
3    -0.7372360235593105  
4    -0.8271626975988577  
..                   ...  
560  -0.8281193006894146  

In [5]:
def getting_day(data_coordinates, year, month, day,hour,minute):
    
    time_coordinates=pd.to_datetime(data_coordinates['Time'])
    #year=2015
    mask = time_coordinates.dt.year == int(year)
    include = data_coordinates[mask]
    time_coordinates2=pd.to_datetime(include['Time'])
    #month='06'
    mask2=time_coordinates2.dt.month == int(month)
    include2 = include[mask2]
    time_coordinates3=pd.to_datetime(include2['Time'])
    #day='07'
    mask3=time_coordinates3.dt.day == int(day)
    include3=include2[mask3]
    time_coordinates4=pd.to_datetime(include3['Time'])
    
    mask4=time_coordinates4.dt.hour == int(hour)
    include4=include3[mask4]
    time_coordinates5=pd.to_datetime(include4['Time'])
    
    mask5=time_coordinates5.dt.minute == int(minute)
    include5=include4[mask5]
    time_coordinates6=pd.to_datetime(include5['Time'])
    
  
    return time_coordinates6, include5

In [6]:
print(getting_day(data,2014,6,14,13,20)[1])

                      Time        x       y       z
152000 2014-06-14 13:20:00  27904.2  169970  161333


In [7]:
print(data)
print(data2)

                       Time                  x                 y  \
0       2014-03-01 00:00:00  -4670965.57270282  1761213.30324318   
1       2014-03-01 00:01:00  -4655896.27124427  1800571.08113835   
2       2014-03-01 00:02:00  -4640495.34888597  1839799.94902527   
3       2014-03-01 00:03:00  -4624763.90816936  1878897.11281282   
4       2014-03-01 00:04:00  -4608703.07517621  1917859.78783793   
...                     ...                ...               ...   
1360796 2016-09-30 23:56:00            1.68484          0.363154   
1360797 2016-09-30 23:57:00            1.68484          0.363154   
1360798 2016-09-30 23:58:00            1.68484          0.363154   
1360799 2016-09-30 23:59:00            1.68484          0.363154   
1360800 2016-10-01 00:00:00            1.68484          0.363154   

                        z  
0        4068774.91931789  
1        4068744.91333393  
2        4068714.90736463  
3        4068684.90141041  
4        4068654.89547039  
...            

In [8]:
list_of_z=[]


for i in np.arange(0,len(data2)):
#for i in np.arange(2,5):
    
    date1=pd.to_datetime(data2['Beginning'][i])
    date2=pd.to_datetime(data2['End'][i])
    #print(type(date1))
    #print(date1)
    
    #filter based on the time column
    mask = (data['Time'] >= date1) & (data['Time'] <= date2)
    df = data[mask]
    df=df.reset_index(drop=True)
    #print('df')
    #print(df)
    #print(df['z'][0])

    list_of_z.append(float(df['z'][0]))
    
    
    

In [9]:
#plt.imshow(np.arange(0,len(hist1[0])),hist1[0])

In [10]:
zdata=data['z'].copy()
zdata=zdata.astype(float)


amount_of_bins=40
rangehist=(-1300,300)
hist1,bins1=np.histogram(list_of_z, bins=amount_of_bins,range=rangehist)
hist2,bins2=np.histogram(zdata,bins=amount_of_bins,range=rangehist)


#NORMALIZED

normalized=[]
error=[]
for i in np.arange(0,len(hist1)):
    normalized.append(hist1[i]/hist2[i])
    error.append(1/hist2[i])
    
y_errormin=error
y_errormax=error
y_error =[y_errormin, y_errormax]    

In [11]:
#plt.figure()
#plt.hist(list_of_z, bins=amount_of_bins ,range=rangehist,color='black')
#plt.xlabel('z (km)')
#plt.ylabel('Amount of MM waves')

#plt.bar(wea,hist1,width=40)

In [12]:
print(len(hist1))
print(len(hist2))
print(hist1)
print(hist2)

40
40
[  0   1   1   0   0   0   0   1   0   0   3   0   0   3   0   0   0   1
   0   0   1   1   0   0   0   3   8  27  11  40  42  84 107  91  90  39
   7   2   0   2]
[   717    715    715    714    713    712    712    711    710    710
    721    719    719    717    718    715    716    715    713    713
    712    712    709   3712   6486   6972   7328  17285   9939  38627
  53047 109794 516566 149242  94219  54525  22916  16303   5152   7764]


In [13]:
#plt.figure()
#plt.hist(zdata, bins=amount_of_bins,range=rangehist,color='black')

#plt.xlabel('z (km)')
#plt.ylabel('Amount of data')

#plt.bar(wea,hist2,width=40)
#plt.bar(bins2[:-1],hist2,width=80)

## total plot

In [14]:
plt.figure()
#plt.hist(normalized, bins=amount_of_bins,range=rangehist,color='black')
plt.xlabel('z (km)')
plt.ylabel('Amount of normalized MM waves')
#plt.plot(bins2[:-1],error,'x')

zxc=[]
for item in bins2[:-1]:
    zxc.append(item+((bins2[1]-bins2[0])/2))

plt.errorbar(zxc, normalized, yerr = y_error, fmt ='o', capsize=4)
plt.bar(zxc,normalized,width=40, color='black')
plt.ylim([0,0.006])
#for i in np.arange(0,len(normalized)):
 #   plt.axvline(bins2[i], c='brown')

(0.0, 0.006)

## Plot with limits

In [15]:
plt.figure()
#plt.hist(normalized, bins=amount_of_bins,range=rangehist,color='black')
plt.xlabel('z (km)')
plt.ylabel('Amount of normalized MM waves')
#plt.plot(bins2[:-1],error,'x')

zxc=[]
for item in bins2[:-1]:
    zxc.append(item+((bins2[1]-bins2[0])/2))

plt.errorbar(zxc, normalized, yerr = y_error, fmt ='o', capsize=4)
plt.bar(zxc,normalized,width=40, color='black')
plt.ylim([0,0.00175])
plt.xlim([-350,350])
#for i in np.arange(0,len(normalized)):
 #   plt.axvline(bins2[i], c='brown')

(-350.0, 350.0)

# What happens with the outgassing?

In [16]:
# Para llevar a cabo este histograma lo que haré será utilizar los dataframe de outgassing y el de coordenadas xyz. 
# Dado que el dataframe de outgassing posee más datos por minuto iré agregando en ese dataframe los valores de la variable
# z del dataframe de coordenadas. Haciendo esto, dado que el dataframe de coordenadas posee solo 1 dato por minuto, habrán
# minutos en los que se repitan datos del valor z, esto indicará que hay mas datos de outgassing por minuto y con lo cual
# será más fácil plotear el histograma dejando en función de z el eje x de éste.

In [17]:
#Importing data

#path="C:/Users/Ariel/Desktop/ESA/position_CK.txt"
#titles=['Time', 'x', 'y', 'z']
#data= pd.read_table(str(path), header=None, names=titles, parse_dates=['Time']) #, sep='\s+'
#data=data.iloc[np.arange(1,len(data)),:] #delete the first row
#data=data.reset_index(drop=True)

#data['Time']= pd.to_datetime(data['Time'])

In [18]:
#-------------------------------------------------------------------------------------------
outgassing="C:/Users/Ariel/Desktop/ESA/Data_GasProduction/gasproduction.txt"

title2=['Timestamp','Neutral gas density','Derived gas production rate']
path= str(outgassing)
data1= pd.read_table(path, header=None, names=title2, sep='\s+', engine='python', parse_dates=['Timestamp'])
 
    
#Filtering
outgassing_copy=data1.copy() 
for j in np.arange(0,len(outgassing_copy)):
    if outgassing_copy['Derived gas production rate'][j]<1000000: 
        outgassing_copy['Timestamp'][j]=np.nan

outgassing_copy=outgassing_copy.dropna()       
data_gas=outgassing_copy.reset_index(drop=True)
data_gas['date'] = pd.to_datetime(data_gas['Timestamp'])

<ipython-input-18-1a523961faef>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outgassing_copy['Timestamp'][j]=np.nan


In [19]:
print(data_gas)
print(data)

                      Timestamp  Neutral gas density  \
0       2014-03-29 16:28:54.480         2.941060e+09   
1       2014-03-29 16:29:52.484         3.666960e+09   
2       2014-03-29 16:31:48.458         3.662410e+09   
3       2014-03-29 16:32:46.461         3.666680e+09   
4       2014-03-29 16:33:44.468         3.656170e+09   
...                         ...                  ...   
1018008 2016-09-30 10:22:30.226         5.476800e+08   
1018009 2016-09-30 10:22:34.233         5.490420e+08   
1018010 2016-09-30 10:22:38.240         5.497850e+08   
1018011 2016-09-30 10:22:42.247         5.515980e+08   
1018012 2016-09-30 10:22:46.254         5.532900e+08   

         Derived gas production rate                    date  
0                       7.468460e+38 2014-03-29 16:28:54.480  
1                       9.311610e+38 2014-03-29 16:29:52.484  
2                       9.299680e+38 2014-03-29 16:31:48.458  
3                       9.310340e+38 2014-03-29 16:32:46.461  
4           

In [20]:
print(len(data))

#SE DEMORARIA 3 HORAS
#SE DEMORARIA 4.4 HORAS EN EL MIO

1360801


In [21]:
#newdataframe=data_gas.copy() #CREAR UN DATAFRAME VACIO
newdataframe= pd.DataFrame()
import warnings
warnings.filterwarnings("ignore")
#for i in np.arange(0,len(data)-1):
for i in np.arange(200000,203300):    
    #print(i)
    date1=pd.to_datetime(data['Time'][i])
    date2=pd.to_datetime(data['Time'][i+1])
    #print(type(date1))
    #print(date1)
    #print(date2)
    
    #filter based on the time column
    mask = (data_gas['date'] >= date1) & (data_gas['date'] <= date2)
    #print(mask)
    df = data_gas[mask]
    #print(df)
    #df=df.reset_index(drop=True)
    
    if len(df)!=0:
        z=[]
        for j in np.arange(0, len(df)):
            z.append(data['z'][i])
        df['z']=z
        newdataframe= pd.concat([newdataframe, df], ignore_index=True)
    #print('df modificado')
    #print(df)
    

In [22]:
print(newdataframe)

                   Timestamp  Neutral gas density  \
0    2014-07-18 02:53:45.240           15259900.0   
1    2014-07-18 02:54:45.245           11927700.0   
2    2014-07-18 02:55:45.250            9695700.0   
3    2014-07-18 02:56:45.255            8247380.0   
4    2014-07-18 02:57:45.260            7028870.0   
...                      ...                  ...   
2883 2014-07-20 04:15:34.732             942924.0   
2884 2014-07-20 04:16:34.746            1006050.0   
2885 2014-07-20 04:17:34.758            1004360.0   
2886 2014-07-20 04:18:34.784            1075600.0   
2887 2014-07-20 04:19:34.804            1008880.0   

      Derived gas production rate                    date            z  
0                    1.039840e+31 2014-07-18 02:53:45.240  5678.171276  
1                    8.126670e+30 2014-07-18 02:54:45.245  5677.782887  
2                    6.605070e+30 2014-07-18 02:55:45.250  5677.394498  
3                    5.617670e+30 2014-07-18 02:56:45.255  5677.006108 

In [23]:
#CAMBIAR EL NUMERO
import csv


newdataframe.to_csv('sasassadasd', index=False, sep='\t')

## Import the new dataframe

In [28]:
outgassing="C:/Users/Ariel/Desktop/ESA/Data_GasProduction/gasproductionwithzcoordinate"

title2=['Timestamp','Neutral gas density','Derived gas production rate','date','z']
path= str(outgassing)
data1= pd.read_table(path, header=None, names=title2 ,sep='\t', engine='python')
data1=data1.iloc[np.arange(1,len(data1)),:] #delete the first row
print(data1)



                       Timestamp Neutral gas density  \
1        2014-03-29 16:28:54.480        2941060000.0   
2        2014-03-29 16:29:52.484        3666960000.0   
3        2014-03-29 16:31:48.458        3662410000.0   
4        2014-03-29 16:32:46.461        3666680000.0   
5        2014-03-29 16:33:44.468        3656170000.0   
...                          ...                 ...   
1018014  2016-09-30 10:22:30.226         547680000.0   
1018015  2016-09-30 10:22:34.233         549042000.0   
1018016  2016-09-30 10:22:38.240         549785000.0   
1018017  2016-09-30 10:22:42.247         551598000.0   
1018018  2016-09-30 10:22:46.254         553290000.0   

        Derived gas production rate                     date                 z  
1                       7.46846e+38  2014-03-29 16:28:54.480  2840780.73958229  
2                       9.31161e+38  2014-03-29 16:29:52.484  2840751.26171685  
3                       9.29968e+38  2014-03-29 16:31:48.458  2840692.30601846  
4  

In [32]:
print(min(data1['z']))

-0.0001717364108793


In [25]:
#zdata=data['z'].copy()
#zdata=zdata.astype(float)


#amount_of_bins=40
#rangehist=(-1300,300)
#hist1,bins1=np.histogram(list_of_z, bins=amount_of_bins,range=rangehist)
#hist2,bins2=np.histogram(zdata,bins=amount_of_bins,range=rangehist)


In [37]:
plt.figure()
amount_of_bins=100
rangehist=(-350,350)
#plt.hist(data1['z'], bins=amount_of_bins,range=rangehist,color='black')
plt.hist(data1['z'].astype(float),range=rangehist ,bins=amount_of_bins,color='black')
plt.xlabel('z (km)')
plt.ylabel('Amount of outgassing data')


#plt.ylim([0,0.006])
#for i in np.arange(0,len(normalized)):
 #   plt.axvline(bins2[i], c='brown')

Text(0, 0.5, 'Amount of outgassing data')